# Exercise: Implementing a Mock EngineCore

Welcome! In this exercise, you will implement a simplified version of `EngineCore`, the central component responsible for coordinating the LLM inference process in a system like vLLM. This will help you understand how requests are received, scheduled, and executed.

By the end of this exercise, you will have:
*   Implemented the `add_request` method to handle incoming generation requests.
*   Implemented the main `step` method to simulate a single iteration of the inference loop.
*   Tracked the state of requests as they move from `PENDING` to `RUNNING` to `FINISHED`.

Let's get started!

In [ ]:
# Setup: Imports and Helper Classes
# Please do not modify this cell.

import enum
from dataclasses import dataclass, field
from typing import List, Dict, Any

# Represents the status of a generation request
class RequestStatus(enum.Enum):
    PENDING = 1
    RUNNING = 2
    FINISHED = 3
    FAILED = 4

# A simple data class to hold information about a request
@dataclass
class Request:
    request_id: str
    prompt: str
    status: RequestStatus = RequestStatus.PENDING
    
    # Allows using Request objects in sets and as dict keys
    def __hash__(self):
        return hash(self.request_id)

# A mock scheduler that decides which requests to run next.
# For this exercise, it simply schedules all pending requests it receives.
class MockScheduler:
    def __init__(self):
        # The log will store the list of requests it was asked to schedule
        self.log: List[List[Request]] = []
        
    def schedule(self, requests: List[Request]) -> List[str]:
        """Schedules requests and returns their IDs."""
        print(f"[Scheduler] Received {len(requests)} pending requests. Scheduling all of them.")
        self.log.append(requests)
        return [req.request_id for req in requests]

# A mock executor that "runs" the model on the scheduled requests.
class MockExecutor:
    def __init__(self):
        # The log will store the list of requests it was asked to execute
        self.log: List[List[Request]] = []
        
    def execute_model(self, requests_to_run: List[Request]) -> None:
        """'Executes' the model on the given requests."""
        print(f"[Executor] Received {len(requests_to_run)} requests to execute.")
        self.log.append(requests_to_run)
        print(f"[Executor] Finished execution for IDs: {[req.request_id for req in requests_to_run]}")

# This is the class you will be working on!
# It contains the core logic for orchestrating the other components.
class EngineCore:
    """A mock EngineCore to manage the lifecycle of inference requests."""
    
    def __init__(self, scheduler: MockScheduler, executor: MockExecutor):
        self.scheduler = scheduler
        self.executor = executor
        self.requests: Dict[str, Request] = {}

    def add_request(self, request_id: str, prompt: str) -> None:
        # This is for you to implement in Exercise 1
        pass

    def step(self) -> None:
        # This is for you to implement in Exercise 2
        pass
        
    def get_request_status(self, request_id: str) -> RequestStatus:
        """Helper to get the status of a specific request."""
        return self.requests[request_id].status

### Exercise 1: Adding a New Request

Your first task is to implement the `add_request` method. This method is the entry point for new prompts coming into the engine.

**Instructions:**
1.  Create a new `Request` object using the `request_id` and `prompt` provided.
2.  The initial status of any new request should be `RequestStatus.PENDING`.
3.  Store this new `Request` object in the `self.requests` dictionary. The `request_id` should be the key.

**Hint:** You can instantiate a `Request` object like this: `Request(request_id=..., prompt=..., status=...)`.

In [ ]:
def add_request_impl(self, request_id: str, prompt: str) -> None:
    """
    Adds a new request to the engine's queue.

    Args:
        request_id: A unique identifier for the request.
        prompt: The input prompt for the language model.
    """


# Monkey-patch the implementation into our class for this exercise
EngineCore.add_request = add_request_impl

In [ ]:
# Test cell for Exercise 1

# 1. Setup
scheduler = MockScheduler()
executor = MockExecutor()
engine = EngineCore(scheduler, executor)
print("Testing Exercise 1: add_request...\n")

# 2. Add two requests
engine.add_request(request_id="req_001", prompt="Hello, world!")
engine.add_request(request_id="req_002", prompt="What is the capital of France?")

# 3. Assertions
# Check if requests were added
assert len(engine.requests) == 2, f"Expected 2 requests in the engine, but got {len(engine.requests)}"
print("✅ Correct number of requests stored.")

# Check request 1 details
req1 = engine.requests.get("req_001")
assert req1 is not None, "Request 'req_001' was not found."
assert req1.prompt == "Hello, world!", f"Incorrect prompt for req_001. Got '{req1.prompt}'"
assert req1.status == RequestStatus.PENDING, f"req_001 should have PENDING status, but got {req1.status}"
print("✅ Request 'req_001' is stored correctly.")

# Check request 2 details
req2 = engine.requests.get("req_002")
assert req2 is not None, "Request 'req_002' was not found."
assert req2.status == RequestStatus.PENDING, f"req_002 should have PENDING status, but got {req2.status}"
print("✅ Request 'req_002' is stored correctly.")

print("\n🎉 All tests for Exercise 1 passed!")

### Exercise 2: Implementing the Core `step` Method

Great job! Now for the most important part: the `step` method. This method represents a single iteration of the engine's main loop. It's responsible for finding pending requests, asking the scheduler what to run, and telling the executor to run them.

**Instructions:**

1.  **Collect Pending Requests:** Create a list of all `Request` objects from `self.requests` that have the status `RequestStatus.PENDING`. If there are none, you can simply `return`.
2.  **Call the Scheduler:** Pass the list of pending requests to `self.scheduler.schedule()`. This will return a list of `request_id`s that have been chosen to run in this step.
3.  **Update Status to RUNNING:** For each `request_id` returned by the scheduler:
    *   Find the corresponding `Request` object.
    *   Change its status to `RequestStatus.RUNNING`.
    *   Add it to a new list, let's call it `requests_to_execute`.
4.  **Call the Executor:** Pass the `requests_to_execute` list to `self.executor.execute_model()`.
5.  **Update Status to FINISHED:** After the executor is called, iterate through the `requests_to_execute` list one last time and change the status of each request to `RequestStatus.FINISHED`. (In a real system, this would happen over multiple steps, but we'll simplify it to a single step for this exercise).

This sequence models the flow of data through the engine's core components.

In [ ]:
def step_impl(self) -> None:
    """
    Performs a single step of the engine's event loop.
    This includes scheduling and executing one batch of requests.
    """


# Monkey-patch the implementation into our class for this exercise
EngineCore.step = step_impl

In [ ]:
# Test cell for Exercise 2

# 1. Setup
scheduler = MockScheduler()
executor = MockExecutor()
engine = EngineCore(scheduler, executor)

engine.add_request(request_id="req_101", prompt="First prompt")
engine.add_request(request_id="req_102", prompt="Second prompt")
engine.add_request(request_id="req_103", prompt="Third prompt")

print("Testing Exercise 2: step...\n")
# 2. Run one step of the engine
engine.step()

# 3. Assertions
# Check scheduler log
assert len(scheduler.log) == 1, f"Scheduler should have been called once, but was called {len(scheduler.log)} times."
scheduled_reqs = scheduler.log[0]
assert len(scheduled_reqs) == 3, f"Scheduler should have received 3 pending requests, but got {len(scheduled_reqs)}."
print("✅ Scheduler was called correctly.")

# Check executor log
assert len(executor.log) == 1, f"Executor should have been called once, but was called {len(executor.log)} times."
executed_reqs = executor.log[0]
assert len(executed_reqs) == 3, f"Executor should have received 3 requests to execute, but got {len(executed_reqs)}."
# Verify that the requests passed to the executor were the ones from the engine
assert set(executed_reqs) == {engine.requests["req_101"], engine.requests["req_102"], engine.requests["req_103"]}, \
    "Executor was not called with the correct set of requests."
print("✅ Executor was called correctly.")

# Check final status of requests
assert engine.get_request_status("req_101") == RequestStatus.FINISHED, "req_101 should be FINISHED."
assert engine.get_request_status("req_102") == RequestStatus.FINISHED, "req_102 should be FINISHED."
assert engine.get_request_status("req_103") == RequestStatus.FINISHED, "req_103 should be FINISHED."
print("✅ All request statuses correctly updated to FINISHED.")

# Check that a second step does nothing
engine.step()
assert len(scheduler.log) == 1, "Scheduler should not be called again when there are no pending requests."
assert len(executor.log) == 1, "Executor should not be called again when there are no pending requests."
print("✅ Engine correctly does nothing on subsequent step.")

print("\n🎉 All tests for Exercise 2 passed!")

### Optional Challenge: Handling a Maximum Batch Size

Well done! For an extra challenge, let's make our `EngineCore` a bit more realistic. Real inference engines can't process an infinite number of requests at once; they are limited by a `max_batch_size`.

**Instructions:**
1. Add a `max_batch_size` parameter to the `EngineCore`'s `__init__` method.
2. Modify your `step` logic: after getting the list of `scheduled_request_ids` from the scheduler, truncate the list so that it contains at most `max_batch_size` IDs.
3. The rest of the logic (updating status, executing, etc.) should only apply to this truncated list of requests.

This ensures that `EngineCore` respects the hardware limits, even if the scheduler suggests a larger batch.

You can modify the `step_impl` function above and re-run the test cells, or create a new `EngineCoreWithBatching` class to test your new logic.

In [ ]:
# Integration Test: Simulating the Engine Over Time

This final cell ties everything together. It simulates a stream of incoming requests and runs the engine's `step` method to process them, showing how the system state evolves over time. No need to write any code here, just run it to see your implementation in action!

print("--- Running Full Integration Test ---")

# 1. Setup with a new engine
scheduler = MockScheduler()
executor = MockExecutor()
engine = EngineCore(scheduler, executor)

# 2. A few requests arrive
print("\n[T=0] Two requests arrive.")
engine.add_request("req_A", "What is vLLM?")
engine.add_request("req_B", "How does PagedAttention work?")

assert engine.get_request_status("req_A") == RequestStatus.PENDING
assert engine.get_request_status("req_B") == RequestStatus.PENDING
print("Initial statuses: PENDING.")

# 3. Engine runs one step
print("\n[T=1] Engine.step() is called.")
engine.step()

assert engine.get_request_status("req_A") == RequestStatus.FINISHED
assert engine.get_request_status("req_B") == RequestStatus.FINISHED
assert len(scheduler.log) == 1
assert len(executor.log) == 1
print("Processed first batch. Statuses are now FINISHED.")

# 4. More requests arrive while the engine is idle
print("\n[T=2] A new request arrives.")
engine.add_request("req_C", "Explain tokenization.")

assert engine.get_request_status("req_C") == RequestStatus.PENDING
print("New request 'req_C' is PENDING.")

# 5. Engine runs another step
print("\n[T=3] Engine.step() is called again.")
engine.step()

assert engine.get_request_status("req_C") == RequestStatus.FINISHED
assert len(scheduler.log) == 2, f"Expected scheduler to be called twice, but got {len(scheduler.log)}"
assert len(executor.log) == 2, f"Expected executor to be called twice, but got {len(executor.log)}"
print("Processed second batch. 'req_C' is now FINISHED.")

# 6. Engine runs one more step when there's nothing to do
print("\n[T=4] Engine.step() is called with no pending requests.")
engine.step()

assert len(scheduler.log) == 2, "Scheduler should not have been called a third time."
assert len(executor.log) == 2, "Executor should not have been called a third time."
print("Engine correctly remained idle.")


print("\n\nCongratulations! You have successfully implemented a mock EngineCore!")
print("You've demonstrated how it adds requests and orchestrates the scheduler and executor to process them.")